# DS2002 · SQL Challenge Set

**Lab — 2026-09-11 · Fall 2026**  

---

## Lab 03 — SQL Challenge Set

Seven questions, one query each. Every query has to produce the right answer when the notebook is run from a fresh kernel, top to bottom.

Two rules that matter as much as getting the answer:

- **Check the row count** against what you expect before you believe a result.
- **Decide what to do about the untagged track and the unplayed tracks.** Several of these questions have a defensible answer either way; what is not defensible is not noticing they exist.

In [1]:
import sqlite3, pandas as pd
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.executescript('''
CREATE TABLE artists (artist_id INTEGER PRIMARY KEY, name TEXT, country TEXT);
CREATE TABLE tracks (track_id INTEGER PRIMARY KEY, title TEXT, artist_id INTEGER, genre TEXT, seconds INTEGER);
CREATE TABLE plays (play_id INTEGER PRIMARY KEY, track_id INTEGER, user TEXT, played_on TEXT);
INSERT INTO artists VALUES
 (1,'Nova Waves','US'),(2,'The Blue Ridge','US'),(3,'Kestrel','UK'),(4,'Marisol','ES');
INSERT INTO tracks VALUES
 (10,'Skyline',1,'Pop',201),(11,'Undertow',1,'Pop',240),(12,'Foothills',2,'Folk',185),
 (13,'Aurora',3,'Electronic',300),(14,'Nightfall',3,'Electronic',275),(15,'Sol',4,'Latin',210),
 (16,'Coastline',2,'Folk',199),(17,'Ridgeline',2,'Folk',225),(18,'Untitled Demo',3,NULL,150);
INSERT INTO plays VALUES
 (100,10,'ava','2026-09-01'),(101,10,'ben','2026-09-01'),(102,13,'ava','2026-09-02'),
 (103,13,'cara','2026-09-02'),(104,14,'ben','2026-09-03'),(105,12,'ava','2026-09-03'),
 (106,15,'dan','2026-09-04'),(107,10,'cara','2026-09-04'),(108,13,'dan','2026-09-05'),
 (109,16,'ava','2026-09-05'),(110,11,'ben','2026-09-06');
''')
conn.commit()

def q(sql):
    return pd.read_sql_query(sql, conn)
print('ready')

ready


### Q1 — Every track with its artist's name and country.

*Expected: 9 rows, one per track.*

In [2]:
q('''
SELECT t.track_id, t.title, a.name AS artist, a.country
FROM tracks t
JOIN artists a ON t.artist_id = a.artist_id
''')
#first row: selecting what columns we want included in the table
#second row: saying to pull from the tracks table
#third row: joining the artists table to the tracks table ON the artist id, so it will look for the artist id in the tracks table and put the matching info it find from the artist table in our printed join table

,track_id,title,artist,country
0,10,Skyline,Nova Waves,US
1,11,Undertow,Nova Waves,US
2,12,Foothills,The Blue Ridge,US
3,13,Aurora,Kestrel,UK
4,14,Nightfall,Kestrel,UK
5,15,Sol,Marisol,ES
6,16,Coastline,The Blue Ridge,US
7,17,Ridgeline,The Blue Ridge,US
8,18,Untitled Demo,Kestrel,UK


**Markdown:** In this table we combined information from two tables into one. To do this we pulled values from the tracks and artists tables as necessary, such as name of artist, country, track id, and title of track. We did this because sometimes it can be more helpful to have all your data in one place versus having to search through multiple tables at once. As you can see all 9 rows are present showing that we correctly created a table of all the tracks with all of the countries and artists listed.

### Q2 — Which genre has the longest average track length?

Return the genre and the average, not just the name.

In [3]:
q('''
SELECT genre, avg(seconds) AS average
FROM tracks
WHERE genre IS NOT NULL
GROUP BY genre
ORDER BY average DESC
LIMIT 1
''')
#Row 1: since we're only looking at the tracks table, we can just name the variables as they are put in the table
#Row 2: again, telling python we're pulling the values from the tracks table
#Row 3: since we are looking at tracks by genre, we do not want to include any tracks that don't have an associated genre, so we say genre can't be NULL or in other words, genre can't be empty
#Row 4: grouping by genre because there could be multiple tracks that belong to the same genre
#Row 5: ordering the table so that the longest average would be the first row, and then descending from there
#Row 6: since we only want the genre that has the longest average track length, we are looking for one answer, therefore we can limit our rows to 1 because the highest average will be the first in the table (due to our ordering), and therefore our answer

,genre,average
0,Electronic,287.5


**Markdown:** For this section I reorganized the data within the tracks table to create new values, like average, as well as reordered the information, in order to find the genre with the longest average track time. I did this because using the data within the table and reordering from there ensures that I am not accidentally using incorrect values, as well as allows me to perform my own analysis of the data for whatever my question may be. For this part specifically, there was printing the genre and average for the longest average time, which happend to be the Electric genre at 287.5 seconds for the average track length.

### Q3 — For each user: how many plays, and how many distinct tracks?

Someone who played one track four times is a different listener from someone who played four different tracks. Your result should make that visible.

In [4]:
q('''
SELECT user,
  COUNT(*) AS plays,
  COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user
''')
#SELECT:looking at user data
#COUNT(*):counts all the rows and each row reports a differnt play so we save that variable name as plays
#COUNT(DISTINCT):counts the unique ones of what variable you tell it, so by doing unique track_id we can see how many are different tracks
#FROM: tells the code to pull data from the plays table
#GROUP and ORDER: group tells the table to put the plays and tracks into categories based on the user while the order by user puts the rows in alphabetical order of user

,user,plays,distinct_tracks
0,ava,4,4
1,ben,3,3
2,cara,2,2
3,dan,2,2


**Markdown:** Here we consolidated our information about plays so that we could get more information about each user. This is valuable because sometimes we want to find patterns not super specific information like what song a user was listening to. By grouping track plays and individual tracks based on each individual, we can see that each time a user listened to a song, they listend to a new distinct track, as seen by the plays matching up with distinct tracks for each person.

### Q4 — Which tracks have never been played?

*Expected: 2 rows.* Hint: `LEFT JOIN` and then keep the rows where the right side came back `NULL`.

In [5]:
q('''
SELECT t.track_id, t.title, COUNT(p.play_id) AS plays
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.track_id IS NULL
GROUP BY t.track_id, t.title
''')
#SELECT: chooses variables we want included in our new table
#FROM: pulling value from the tracks table
#LEFT JOIN: by doing this, we included tracks that both have a play value and don't, so we can see which tracks haven't been play
#WHERE: null means we are only looking for track ids with no plays since null means there is no associated value, because we don't want an associate play
#GROUP: telling the code to put the track_id and title together

,track_id,title,plays
0,17,Ridgeline,0
1,18,Untitled Demo,0


**Markdown:** Here we left joined data to purposely keep in rows that had empty values, in this case no plays, so that we could analyze which songs were being listend to and which weren't. We used a left join because the typical join is an inner join which only returns rows with matches, since the songs we wanted to see had no plays, there would not have been any matching track_id data in the plays table, and therefore the values we were looking for of the unplayed tracks would not have been included. Now with the left join and limitation that we only wanted data that had no value of plays, we can see tracks that have been played zero times.

### Q5 — Rank artists by total listening time.

Sum the seconds actually listened across all plays, most to least, and include a minutes column rounded to one decimal.

In [6]:
q('''
SELECT a.name AS artist, sum(t.seconds) AS total_seconds_listened, ROUND(sum(t.seconds)/60.0, 1) AS minutes_listened
FROM plays p
JOIN tracks t ON p.track_id = t.track_id
JOIN artists a ON t.artist_id = a.artist_id
GROUP BY a.artist_id, a.name
ORDER BY total_seconds_listened DESC
''')
#SELECT: we want to pull the key values we want for our columns, artist name, the total seconds listened to across plays and that number's equivalent in minutes
#FROM: since we're looking at time across plays, we want to pull from the plays table
#JOIN: first we join tracks between plays and tracks table so we can get the correct times for the plays/tracks, then we join from artists table so that we can see which tracks belong to each artist, allowing us to find the artist's listening time
#GROUP: putting the tracks together by who the arist is
#ORDER DESC: this puts the seconds from most to least as asked for in the directions

,artist,total_seconds_listened,minutes_listened
0,Kestrel,1175,19.6
1,Nova Waves,843,14.1
2,The Blue Ridge,384,6.4
3,Marisol,210,3.5


**Markdown:** For this question I combined all three tables in order to combines plays for each track with their times and then sort by artist. This is useful because sometimes you need to pull data from more than just two tables, and by making sure you do so in the right order, with the correct primary keys, you can make analysis much easier. I was also able to manipulate the values into new variables by summing and dividing to find that Kestrel is the artist with the most seconds and therefore most minutes listened to.

### Q6 — Which tracks are missing a genre?

Return the track id and title. Then, in a comment, say what `WHERE genre != 'Pop'` would have done to these rows and why.

In [7]:
q('''
SELECT track_id, title
FROM tracks
WHERE genre IS NULL
''')
#When you look at the track id and title for a track that is missing a genre, or in other words, genre is null, you see it is track 18, Untitled Demo
#If you had used WHERE genre != 'Pop' you would have dropped the tracks that are missing a genre because it asked you for genres that are not pop, to which the code would see the missing genre as an unknown value rather than a non-pop value, therefore it would not have included the tracks with missing genres

,track_id,title
0,18,Untitled Demo


**Markdown:** The question asks for which track is missing a genre, so we look at the tracks table and see where there is a track with a null, or empty value, for genre. The reason other options wouldn't work, such as the WHERE genre != 'Pop' code is because of the way the code reads an empty value. When looking for something to be not Pop, the code is looking for values that don't match Pop. However, no data or an empty genre isn't considered its own category, but rather incomplete and therefore unknown. Since it is unknown, it cannot be confirmed that the no genre track is not a pop genre, therefore the track would not be a row in the table because the code can't verify that the WHERE genre != 'Pop'statement would be true.

### Q7 — Plays per day.

`played_on` is stored as text like `'2026-09-01'`. Count plays per date, earliest first, and include the number of distinct users active that day.

In [8]:
q('''
SELECT played_on, COUNT(*) AS plays, COUNT(DISTINCT user) AS distinct_users
FROM plays
GROUP BY played_on
ORDER BY played_on ASC
''')
#SELECT: selecting coulmns, we want the dates played on, and then for each of those dates, how many plays and how many distinct users
#FROM: pulling from the plays table
#GROUP: we want the plays grouped by data and distinct users grouped by date so we used the date variable, played_on
#ORDER: order by ascending means the table will display the earliest date first and then continue on

,played_on,plays,distinct_users
0,2026-09-01,2,2
1,2026-09-02,2,2
2,2026-09-03,2,2
3,2026-09-04,2,2
4,2026-09-05,2,2
5,2026-09-06,1,1


**Markdown:** I organized the plays table data so that we could concentrate on what happened each day. This can be helpful to ascertain patterns over the course of time. I did this by grouping by date so that all the necessary information would be presented on daily timelines as opposed to user based. This came back with plays and distinct users being 2 for each everyday except for the last day, 2026-09-06.

### Validate your work

**TODO:** uncomment these and make them pass. Assign your query results to the variables as you go — for example `q1 = q('''...''')`.

In [9]:
q1 = q('''
SELECT t.track_id, t.title, a.name AS artist, a.country
FROM tracks t
JOIN artists a ON t.artist_id = a.artist_id
''')

q4 = q('''
SELECT t.track_id, t.title, COUNT(p.play_id) AS plays
FROM tracks t
LEFT JOIN plays p ON t.track_id = p.track_id
WHERE p.track_id IS NULL
GROUP BY t.track_id, t.title
''')

q3 = q('''
SELECT user,
  COUNT(*) AS plays,
  COUNT(DISTINCT track_id) AS distinct_tracks
FROM plays
GROUP BY user
ORDER BY user
''')

assert len(q1) == 9, 'Q1 should return one row per track'
assert len(q4) == 2, 'Q4: two tracks have never been played'
assert q3['plays'].sum() == 11, 'Q3 should account for all 11 plays'
print('checks passed.')

checks passed.


### Write-up

Pick the query that gave you the most trouble and explain what you had wrong before you had it right. Name the specific misunderstanding — "I put the aggregate in WHERE" or "I used an inner join and lost the tracks with no plays" — not "it was confusing."

**My Answer:** The hardest question for me was query 4 because I didn't understand the left join versus right side that was in the hint, so I originally used the inner join which I had grown confident using. This unfortunately got me the wrong information since the two tracks that hadn't been played had no match on the plays table, therefore completely cutting them out of my new table. However, I then looked back at the hint and my notes to be able to get the correct answer. Since the left join would keep every track on the orginal table (or the left side), when I joined it to the new table, it would fill in everything else respectively on the right side. This meant that even null information would be included, which is what I needed since the data I was relying on would not have appeared on the plays table, therefore producing a zero for the plays. Then I also got confused with trying to just print the null values because my original thought was that I needed LIMIT to limit the responses to those 2 rows, but I quickly realized that LIMIT doesn't allow you to filter which rows are printed, it just prints the first 2. From here I realized I should switch from LIMIT to WHERE in order to put which rows should be kept, in this case the null tracks with zero plays.